In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
from pathlib import Path
from os import environ


# IN_COLAB = False
# if "DRIVE_HOME" in environ:
  # ROOT = Path(f"{environ.get("DRIVE_HOME")}/colab/outputs/waterloo-slt-reading-group")
# else:
ROOT = Path(f"{Path.cwd().parents[1]}/outputs")
basedir = Path(f"{ROOT}/mixture/binom2d")
datadir = Path(f"{basedir}/data")
outputdir = Path(f"{basedir}/wbic-bias")

if not outputdir.exists():
  outputdir.mkdir(exist_ok=True)
  print(f"Created {outputdir}!")

print(f"Using datadir={datadir}")
print(f"Using outputdir={outputdir}")

Using datadir=/home/ubuntu/code/waterloo-slt-reading-group/zoo/python/projects/luna/outputs/mixture/binom2d/data
Using outputdir=/home/ubuntu/code/waterloo-slt-reading-group/zoo/python/projects/luna/outputs/mixture/binom2d/wbic-bias


In [3]:
import pandas as pd

dgps_file = f"{datadir}/dgp.csv"
dgps=pd.read_csv(dgps_file, index_col=0).reset_index()
dgps.head()

,dsid,p0,p1,w0,w1
0,regular,0.15,0.35,0.75,0.25
1,e-singular,0.25,0.35,0.75,0.25
2,singular1,0.15,0.35,1.00,0.00
3,singular2,0.35,0.35,0.75,0.25


In [4]:
from sklearn_extensions.mixbinom import BinomialMixture
import numpy as np


def find_truth_by_dsid(dsid: str):
  dgp = dgps.query(f"dsid=='{dsid}'")
  truth = dgp[["p0", "p1", "w0", "w1"]].iloc[0].tolist()
  return truth

def rlct_by_dsid(dsid: str):
  truth = find_truth_by_dsid(dsid)
  n_components = np.ceil(len(truth)/2)
  rlct = None
  match dsid:
    case "regular" | "e-singular":
      rlct = (n_components*2-1)/2
    case "singular1" | "singular2":
      rlct = 1
    case _:
      raise Exception(f"Uknown dsid={dsid}")

  return rlct

def approx_free_energy_by_dsid(dsid, n_trials, X):
  n=len(X)
  
  average_log_likelihood = None
  model = BinomialMixture(n_components=2, n_trials=n_trials, enforce_ordering=False)
  input_data = np.column_stack([X, np.full_like(X, n_trials)])
  model.fit(input_data)
  mle, _ = model.point_estimate()
  log_p = mixbinom.logpmf(weights=[mle[2], 1-mle[2]], probs=[mle[0], mle[1]], n=n_trials, x=X) # sample likelihood under the mle
  average_log_likelihood = log_p.mean()

  afe = -n*average_log_likelihood+rlct_by_dsid(dsid)*np.log(n)
  return afe


def expected_free_energy_by_dsid(dsid, n_trials, n):
  X = np.arange(0, n_trials + 1)  # 0 to n_trials inclusive
  truth = find_truth_by_dsid(dsid)
  
  probs = truth[0:2]    # [p0, p1]
  weights = truth[2:4]  # [w0, w1]
  
  log_q = mixbinom.logpmf(n=n_trials, weights=weights, probs=probs, x=X)
  
  expected_log_likelihood = np.sum(np.exp(log_q) * log_q)
  
  second_order_term = rlct_by_dsid(dsid) * np.log(n)
  efe = -n * expected_log_likelihood + second_order_term
  
  return efe

In [5]:
# compute AFE and WBIC for many samples, 
# AFE requires sampling with beta=1 while WBIC requires sampling with beta=1/sqrt(n)
# for different n

In [6]:
import pandas as pd
import json
from pathlib import Path

results_file = Path(f"{outputdir}/fe_estimators_results.csv")

# Load existing results if file exists
if results_file.exists():
  fe_estimators_df = pd.read_csv(results_file)
  # Create a set of completed (run, regime, dsid) tuples for fast lookup
  completed = set(
      zip(fe_estimators_df["trial"], fe_estimators_df["regime"], fe_estimators_df["dsid"])
  )
  fe_estimators_data = fe_estimators_df.to_dict("records")
else:
  completed = set()
  fe_estimators_data = []

def save_results():
  """Save current results to disk."""
  pd.DataFrame(fe_estimators_data).to_csv(results_file, index=False)

In [7]:
from joblib import Parallel, delayed
from tqdm import tqdm
from itertools import product
import time
from pymc_extensions.tempered_mixbinom import TemperedBinomialMixture
from pymc_extensions import pmx
from scipy_extensions import mixbinom
from tqdm.notebook import tqdm
import pymc as pm
import numpy as np
import arviz as az

def run_single_dsid(dsid, run, regime, datadir, n_trials, n_draws, n_tune, n_chains):
    dataset = pd.read_csv(f"{datadir}/{dsid}-{regime}.csv")
    X = dataset.iloc[:, run].to_numpy()
    n_obs = len(X)
    
    efe = expected_free_energy_by_dsid(dsid=dsid, n_trials=n_trials, n=n_obs)
    afe = approx_free_energy_by_dsid(dsid=dsid, n_trials=n_trials, X=X)
    
    with TemperedBinomialMixture(X=X, n_trials=100, beta=1/np.log(n_obs)) as model:
        idata = model.sample(
            draws=n_draws,
            tune=n_tune,
            chains=n_chains,
            progressbar=False,
            nuts_sampler="nutpie",
            cores=2
        )
        
        diverging = idata.sample_stats.diverging.values
        divs_per_chain = diverging.sum(axis=1)
        
        weights = pmx.column_stack_vars(idata, ["weights"])
        probs = pmx.column_stack_vars(idata, ["p0", "p1"])
        log_likelihood = mixbinom.log_likelihood(weights, probs, n=n_trials, x=X)
        wbic = -log_likelihood.mean()
        
        print(f"run={run}, regime={regime}, dsid={dsid}, wbic={wbic:.4f}, afe={afe:.4f}, efe={efe:.4f}")
        
        return {
            "dsid": dsid,
            "regime": regime,
            "n": regime,
            "trial": run,
            "wbic": wbic,
            "afe": afe,
            "efe": efe,
            "chains": n_chains,
            "draws": n_draws,
            "tune": n_tune,
            "mean_divergences": divs_per_chain.mean(),
            "total_divergences": diverging.sum(),
            "max_divergences": divs_per_chain.max(),
            "divergences_per_chain": divs_per_chain.tolist(),
            "chain_tree_depth": idata.sample_stats.depth.values.max()
        }

n_components = 2
n_trials=100
regimes = [50, 250, 5000]

# mcmc settings
n_tune=2000
n_draws=2000
n_chains=2

# All dsids
all_dsids = dgps["dsid"].unique()

# Main loop - one run at a time, parallelize all (regime, dsid) combos
for run in tqdm(range(1000), desc="runs"):
  start = time.perf_counter()
  # Build list of all (regime, dsid) pairs that haven't been completed
  tasks_to_run = [
    (regime, dsid) 
    for regime, dsid in product(regimes, all_dsids)
    if (run, regime, dsid) not in completed
  ]
  
  if not tasks_to_run:
      continue
  
  # Run all regime x dsid combinations in parallel
  results = Parallel(n_jobs=4, verbose=10)(
    delayed(run_single_dsid)(
        dsid, run, regime, datadir, n_trials, n_draws, n_tune, n_chains
    )
    for regime, dsid in tasks_to_run
  )
  
  # Collect results
  for result in results:
    fe_estimators_data.append(result)
    completed.add((result["trial"], result["regime"], result["dsid"]))

  end = time.perf_counter()
  # Save after each run completes
  save_results()
  print(f"run {run} in {start-end:0.3f}")
  !git add "../../outputs/mixture/binom2d/wbic-bias/fe_estimators_results.csv"
  !git commit -m "run {run} complete"
  !git push

runs:   0%|          | 0/1000 [00:00<?, ?it/s]

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.


/home/ubuntu/code/waterloo-slt-reading-group/zoo/python/libs/scipy_extensions/src/scipy_extensions/mixbinom.py:21: RuntimeWarning: divide by zero encountered in log
  return logsumexp(result + np.log(weights), axis=1)


/home/ubuntu/code/waterloo-slt-reading-group/zoo/python/libs/scipy_extensions/src/scipy_extensions/mixbinom.py:21: RuntimeWarning: divide by zero encountered in log
  return logsumexp(result + np.log(weights), axis=1)


[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:   37.6s


...

[Parallel(n_jobs=4)]: Done   7 out of  12 | elapsed:   39.4s remaining:   28.2s


/home/ubuntu/code/waterloo-slt-reading-group/zoo/python/libs/scipy_extensions/src/scipy_extensions/mixbinom.py:21: RuntimeWarning: divide by zero encountered in log
  return logsumexp(result + np.log(weights), axis=1)


[Parallel(n_jobs=4)]: Done   9 out of  12 | elapsed:  1.0min remaining:   20.6s


run=33, regime=250, dsid=e-singular, wbic=805.1802, afe=804.8401, efe=810.5629
run=33, regime=5000, dsid=e-singular, wbic=16012.6656, afe=16012.3481, efe=16058.3906


KeyboardInterrupt: 

In [ ]:
# from pymc_extensions.tempered_mixbinom import TemperedBinomialMixture
# from pymc_extensions import pmx
# from scipy_extensions import mixbinom
# from tqdm.notebook import tqdm
# import pymc as pm
# import numpy as np
# import arviz as az


# n_components = 2
# n_trials=100
# regimes = [50, 250, 5000]

# # mcmc settings
# n_tune=2000
# n_draws=2000
# n_chains=2

# # read all the data so we can nicely loop 
# for run in tqdm(range(1000), desc=f"runs "):
#   for regime in tqdm(regimes, desc="regimes"):
#     # Filter to only incomplete dsids
#     dsids_to_run = [
#       dsid for dsid in dgps["dsid"].unique()
#       if (run, regime, dsid) not in completed
#     ]
    
#     # Run in parallel
#     results = Parallel(n_jobs=8)(  # adjust based on your vCPUs
#       delayed(run_single_dsid)(
#           dsid, run, regime, datadir, dgps, n_trials, n_draws, n_tune, n_chains
#       )
#       for dsid in dsids_to_run
#     )
    
#     # Collect results
#     for result in results:
#       fe_estimators_data.append(result)
#       completed.add((result["trial"], result["regime"], result["dsid"]))
#     # for dsid in dgps["dsid"].unique():
#       # Skip if already completed
#       # if (run, regime, dsid) in completed:
#       #     continue

#       # # try:
#       # dataset = pd.read_csv(f"{datadir}/{dsid}-{regime}.csv")

#       # X = dataset.iloc[:, run].to_numpy()
#       # n_obs = len(X)
#       # efe = expected_free_energy_by_dsid(dsid=dsid, n_trials=n_trials, n=n_obs)
#       # afe = approx_free_energy_by_dsid(dsid=dsid, n_trials=n_trials, X=X)

#       # with TemperedBinomialMixture(X=X, n_trials=100, beta=1/np.log(n_obs)) as model:
#       #   idata = model.sample(draws=n_draws, 
#       #                        tune=n_tune, 
#       #                        chains=n_chains, 
#       #                        progressbar=False,
#       #                        nuts_sampler="nutpie",
#       #                        cores=2) # running on ec2-c6i

#       #   # Divergences are stored in sample_stats as a boolean array (chain, draw)
#       #   diverging = idata.sample_stats.diverging.values
        
#       #   # Divergences per chain
#       #   divs_per_chain = diverging.sum(axis=1)  # array with one value per chain
        
#       #   # Summary statistics
#       #   mean_divergences = divs_per_chain.mean()
#       #   total_divergences = diverging.sum()
#       #   max_divergences = divs_per_chain.max()
        
#       #   weights = pmx.column_stack_vars(idata, ["weights"])
#       #   probs = pmx.column_stack_vars(idata, ["p0", "p1"])
#       #   log_likelihood = mixbinom.log_likelihood(weights, probs, n=n_trials, x=X)
#       #   wbic = -log_likelihood.mean()
#       #   print(f"run={run}, regime={regime}, dsid={dsid},wbic={wbic}, afe={afe}, efe={efe}")
#       #   result = {
#       #     "dsid": dsid,
#       #     "regime": regime,
#       #     "n": regime,
#       #     "trial": run,
#       #     "wbic": wbic,
#       #     "afe": afe,
#       #     "efe": efe,
#       #     "chains": n_chains,
#       #     "draws": n_draws,
#       #     "tune": n_tune,
#       #     "mean_divergences": mean_divergences,
#       #     "total_divergences": total_divergences,
#       #     "max_divergences": max_divergences,
#       #     "divergences_per_chain": divs_per_chain.tolist(),
#       #     "chain_tree_depth": idata.sample_stats.depth.values.max()
#       #   }

#       #   fe_estimators_data.append(result)
#       #   completed.add((run, regime, dsid))
        
#   # Save after each successful run
#   save_results()
#   !git add "../../outputs/mixture/binom2d/wbic-bias/fe_estimators_results.csv"
#   !git commit -m "more runs"
#   !git push

In [ ]:
print(idata.sample_stats)

In [ ]:
fe_estimates_df = pd.DataFrame(fe_estimators_data)
fe_estimates_df.style.format({'wbic': '{:.3f}', 'efe': '{:.3f}'})
fe_estimates_df.head()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Compute the differences
fe_estimates_df['wbic_minus_afe'] = fe_estimates_df['wbic'] - fe_estimates_df['afe']
fe_estimates_df['wbic_minus_efe'] = fe_estimates_df['wbic'] - fe_estimates_df['efe']

# Summary statistics
summary = fe_estimates_df.groupby(['dsid', 'n']).agg(
    wbic_afe_mean=('wbic_minus_afe', 'mean'),
    wbic_afe_std=('wbic_minus_afe', 'std'),
    wbic_efe_mean=('wbic_minus_efe', 'mean'),
    wbic_efe_std=('wbic_minus_efe', 'std'),
).reset_index()

# Get RLCT for each dsid
dsids = fe_estimates_df['dsid'].unique()
n_range = np.linspace(fe_estimates_df['n'].min(), fe_estimates_df['n'].max(), 100)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: Bias (Mean of WBIC - EFE) ~ λ log(n)
ax = axes[0]
for dsid in dsids:
    data = summary[summary['dsid'] == dsid]
    ax.plot(data['n'], data['wbic_efe_mean'], 'o-', label=dsid, markersize=8)
    
    # Theoretical reference: λ log(n) anchored at first point
    lam = rlct_by_dsid(dsid)
    n0 = data['n'].iloc[0]
    offset = data['wbic_efe_mean'].iloc[0] - lam * np.log(n0)
    ax.plot(n_range, lam * np.log(n_range) + offset, '--', alpha=0.5)

ax.set_xscale('log')
ax.set_xlabel('n')
ax.set_ylabel('Mean(WBIC − EFE)')
ax.set_title(r'Bias: $\mathbb{E}[\mathrm{WBIC} - nS] \approx \lambda \log n$')
ax.legend()

# Right: Fluctuation (Std of WBIC - AFE) ~ √log(n)
ax = axes[1]
for dsid in dsids:
    data = summary[summary['dsid'] == dsid]
    ax.plot(data['n'], data['wbic_afe_std'], 'o-', label=dsid, markersize=8)

# Reference lines
n0 = summary['n'].min()
std0 = summary[summary['n'] == n0]['wbic_afe_std'].median()

ax.plot(n_range, np.full_like(n_range, std0), '--', color='gray', linewidth=2, label='O(1)')
c = std0 / np.sqrt(np.log(n0))
ax.plot(n_range, c * np.sqrt(np.log(n_range)), ':', color='gray', linewidth=2, label=r'O($\sqrt{\log n}$)')

ax.set_xscale('log')
ax.set_xlabel('n')
ax.set_ylabel('Std(WBIC − AFE)')
ax.set_title(r'Fluctuation: $\mathrm{Std}(\mathrm{WBIC}) \sim O(\sqrt{\log n})$')
ax.legend()

plt.suptitle('WBIC Diverges from True Free Energy', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: Bias / log(n) → λ
ax = axes[0]
for dsid in dsids:
    data = summary[summary['dsid'] == dsid]
    normalized = data['wbic_efe_mean'] / np.log(data['n'])
    lam = rlct_by_dsid(dsid)
    ax.plot(data['n'], normalized, 'o-', label=f'{dsid} (λ={lam})', markersize=8)
    ax.axhline(lam, linestyle='--', alpha=0.3)

ax.set_xscale('log')
ax.set_xlabel('n')
ax.set_ylabel(r'$\frac{\mathrm{Mean(WBIC - EFE)}}{\log n}$')
ax.set_title(r'Should converge to $\lambda$')
ax.legend()

# Right: Std / √log(n) → constant
ax = axes[1]
for dsid in dsids:
    data = summary[summary['dsid'] == dsid]
    normalized = data['wbic_afe_std'] / np.sqrt(np.log(data['n']))
    ax.plot(data['n'], normalized, 'o-', label=dsid, markersize=8)

ax.set_xscale('log')
ax.set_xlabel('n')
ax.set_ylabel(r'$\frac{\mathrm{Std(WBIC - AFE)}}{\sqrt{\log n}}$')
ax.set_title('Should be constant (for singular)')
ax.legend()

plt.suptitle('Normalized: Confirming Divergence Rates', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Compute the differences
fe_estimates_df['wbic_minus_efe'] = fe_estimates_df['wbic'] - fe_estimates_df['efe']

# Summary statistics
summary = fe_estimates_df.groupby(['dsid', 'n']).agg(
    wbic_efe_mean=('wbic_minus_efe', 'mean'),
).reset_index()

dsids = fe_estimates_df['dsid'].unique()
n_range = np.linspace(fe_estimates_df['n'].min(), fe_estimates_df['n'].max(), 100)

# Get RLCT values
rlct_regular = rlct_by_dsid('regular')
rlct_singular = rlct_by_dsid('singular1')  # or whichever singular dsid you want as reference

fig, ax = plt.subplots(figsize=(10, 6))

# Plot data for each dsid
for dsid in dsids:
    data = summary[summary['dsid'] == dsid]
    ax.plot(data['n'], data['wbic_efe_mean'], 'o-', label=dsid, markersize=8)

# Reference line for regular: λ_regular * log(n)
data_reg = summary[summary['dsid'] == 'regular']
n0 = data_reg['n'].iloc[0]
offset_reg = data_reg['wbic_efe_mean'].iloc[0] - rlct_regular * np.log(n0)
ax.plot(n_range, rlct_regular * np.log(n_range) + offset_reg, '--', 
        color='gray', linewidth=2, label=f'λ={rlct_regular} log(n) (regular)')

# Reference line for singular: λ_singular * log(n)
data_sing = summary[summary['dsid'] == 'singular1']
n0 = data_sing['n'].iloc[0]
offset_sing = data_sing['wbic_efe_mean'].iloc[0] - rlct_singular * np.log(n0)
ax.plot(n_range, rlct_singular * np.log(n_range) + offset_sing, ':', 
        color='gray', linewidth=2, label=f'λ={rlct_singular} log(n) (singular)')

ax.set_xscale('log')
ax.set_xlabel('n')
ax.set_ylabel('Mean(WBIC − EFE)')
ax.set_title(r'WBIC Bias: $\mathbb{E}[\mathrm{WBIC} - nS] \approx \lambda \log n$')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, dsid in zip(axes, dsids):
    data = summary[summary['dsid'] == dsid]
    lam = rlct_by_dsid(dsid)
    
    # Plot bias
    ax.plot(data['n'], data['wbic_efe_mean'], 'o-', color='C0', label='Mean(WBIC − EFE)', markersize=8)
    
    # Theoretical λ log(n)
    n0 = data['n'].iloc[0]
    offset = data['wbic_efe_mean'].iloc[0] - lam * np.log(n0)
    ax.plot(n_range, lam * np.log(n_range) + offset, '--', color='C0', alpha=0.5, label=f'λ log(n), λ={lam}')
    
    ax.set_xscale('log')
    ax.set_xlabel('n')
    ax.set_ylabel('Mean(WBIC − EFE)')
    ax.set_title(f'{dsid} (λ = {lam})')
    ax.legend()

plt.suptitle('WBIC Bias by Data Generating Process', y=1.02)
plt.tight_layout()
plt.show()